In [6]:
import gymnasium as gym

print(gym.__version__)

1.3.0


In [7]:
import gymnasium as gym

In [8]:
env = gym.make("CartPole-v1")

In [10]:
%pip install tyro


   ---------------------------------------- 0/3 [typeguard]
   -------------------------- ------------- 2/3 [tyro]
   -------------------------- ------------- 2/3 [tyro]
   -------------------------- ------------- 2/3 [tyro]
   -------------------------- ------------- 2/3 [tyro]
   ---------------------------------------- 3/3 [tyro]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
%pip install tensorboard

   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.5 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.5 MB 1.0 MB/s eta 0:00:05
   ----- ---------------------------------- 0.8/5.5 MB 1.3 MB/s eta 0:00:04
   ------- -------------------------------- 1.0/5.5 MB 1.3 MB/s eta 0:00:04
   --------- ------------------------------ 1.3/5.5 MB 1.4 MB/s eta 0:00:03
   ----------- ---------------------------- 1.6/5.5 MB 1.3 MB/s eta 0:00:04
   ------------- -------------------------- 1.8/5.5 MB 1.3 MB/s eta 0:00:03
   --------------- ------------------------ 2.1/5.5 MB 1.3 MB/s eta 0:00:03
   ----------------- ---------------------- 2.4/5.5 MB 1.2 MB/s eta 0:00:03
   ------------------- -------------------- 2.6/5.5 MB 1.3 MB/s eta 0:00:03
   -------------------- ------------------- 2.9/5.5 MB 1.3 MB/s eta 0:00:02
   ------------------------ -----


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# ============================================================
# PPO - CleanRL style - Jupyter Notebook version
# CartPole-v1
# ============================================================

import os
import random
import time
from dataclasses import dataclass

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter


# ============================================================
# 1. Arguments
# ============================================================

@dataclass
class Args:
    exp_name: str = "ppo"
    seed: int = 1
    torch_deterministic: bool = True
    cuda: bool = True

    track: bool = False
    wandb_project_name: str = "cleanRL"
    wandb_entity: str = None

    capture_video: bool = False

    # Environment
    env_id: str = "CartPole-v1"

    # Training
    total_timesteps: int = 10000
    learning_rate: float = 2.5e-4

    num_envs: int = 4
    num_steps: int = 128

    anneal_lr: bool = True

    # PPO
    gamma: float = 0.99
    gae_lambda: float = 0.95

    num_minibatches: int = 4
    update_epochs: int = 4

    norm_adv: bool = True

    clip_coef: float = 0.2
    clip_vloss: bool = True

    ent_coef: float = 0.01
    vf_coef: float = 0.5

    max_grad_norm: float = 0.5

    target_kl: float = None

    # Runtime values
    batch_size: int = 0
    minibatch_size: int = 0
    num_iterations: int = 0


args = Args()

# Calculate runtime parameters
args.batch_size = args.num_envs * args.num_steps
args.minibatch_size = args.batch_size // args.num_minibatches
args.num_iterations = args.total_timesteps // args.batch_size

print("Batch size:", args.batch_size)
print("Minibatch size:", args.minibatch_size)
print("Number of iterations:", args.num_iterations)


# ============================================================
# 2. Environment
# ============================================================

def make_env(env_id, idx, capture_video, run_name):

    def thunk():

        if capture_video and idx == 0:

            env = gym.make(
                env_id,
                render_mode="rgb_array"
            )

            env = gym.wrappers.RecordVideo(
                env,
                f"videos/{run_name}"
            )

        else:

            env = gym.make(env_id)

        env = gym.wrappers.RecordEpisodeStatistics(env)

        return env

    return thunk


# ============================================================
# 3. Neural network initialization
# ============================================================

def layer_init(
    layer,
    std=np.sqrt(2),
    bias_const=0.0
):

    torch.nn.init.orthogonal_(
        layer.weight,
        std
    )

    torch.nn.init.constant_(
        layer.bias,
        bias_const
    )

    return layer


# ============================================================
# 4. PPO Agent
# ============================================================

class Agent(nn.Module):

    def __init__(self, envs):

        super().__init__()

        # ----------------------------
        # Critic
        # ----------------------------

        self.critic = nn.Sequential(

            layer_init(
                nn.Linear(
                    np.array(
                        envs.single_observation_space.shape
                    ).prod(),
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 64)
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 1),
                std=1.0
            )
        )

        # ----------------------------
        # Actor
        # ----------------------------

        self.actor = nn.Sequential(

            layer_init(
                nn.Linear(
                    np.array(
                        envs.single_observation_space.shape
                    ).prod(),
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 64)
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(
                    64,
                    envs.single_action_space.n
                ),
                std=0.01
            )
        )

    # ----------------------------
    # Critic output
    # ----------------------------

    def get_value(self, x):

        return self.critic(x)

    # ----------------------------
    # Actor + Critic
    # ----------------------------

    def get_action_and_value(
        self,
        x,
        action=None
    ):

        logits = self.actor(x)

        probs = Categorical(
            logits=logits
        )

        if action is None:

            action = probs.sample()

        return (
            action,
            probs.log_prob(action),
            probs.entropy(),
            self.critic(x)
        )


# ============================================================
# 5. Run name
# ============================================================

run_name = (
    f"{args.env_id}__"
    f"{args.exp_name}__"
    f"{args.seed}__"
    f"{int(time.time())}"
)


# ============================================================
# 6. TensorBoard
# ============================================================

writer = SummaryWriter(
    f"runs/{run_name}"
)

writer.add_text(
    "hyperparameters",
    "|param|value|\n|-|-|\n%s"
    %
    (
        "\n".join(
            [
                f"|{key}|{value}|"
                for key, value in vars(args).items()
            ]
        )
    )
)


# ============================================================
# 7. Random seeds
# ============================================================

random.seed(args.seed)

np.random.seed(args.seed)

torch.manual_seed(args.seed)

torch.backends.cudnn.deterministic = (
    args.torch_deterministic
)


# ============================================================
# 8. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available() and args.cuda
    else "cpu"
)

print("Device:", device)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 9. Create environments
# ============================================================

envs = gym.vector.SyncVectorEnv(

    [
        make_env(
            args.env_id,
            i,
            args.capture_video,
            run_name
        )

        for i in range(args.num_envs)
    ]
)


assert isinstance(
    envs.single_action_space,
    gym.spaces.Discrete
)


# ============================================================
# 10. Create agent
# ============================================================

agent = Agent(envs).to(device)


optimizer = optim.Adam(
    agent.parameters(),
    lr=args.learning_rate,
    eps=1e-5
)


# ============================================================
# 11. PPO storage
# ============================================================

obs = torch.zeros(

    (
        args.num_steps,
        args.num_envs
    )
    + envs.single_observation_space.shape

).to(device)


actions = torch.zeros(

    (
        args.num_steps,
        args.num_envs
    )
    + envs.single_action_space.shape

).to(device)


logprobs = torch.zeros(

    args.num_steps,
    args.num_envs

).to(device)


rewards = torch.zeros(

    args.num_steps,
    args.num_envs

).to(device)


dones = torch.zeros(

    args.num_steps,
    args.num_envs

).to(device)


values = torch.zeros(

    args.num_steps,
    args.num_envs

).to(device)


# ============================================================
# 12. Reset environment
# ============================================================

global_step = 0

start_time = time.time()

next_obs, _ = envs.reset(
    seed=args.seed
)

next_obs = torch.Tensor(
    next_obs
).to(device)


next_done = torch.zeros(
    args.num_envs
).to(device)


# ============================================================
# 13. PPO Training Loop
# ============================================================

for iteration in range(
    1,
    args.num_iterations + 1
):

    # ========================================================
    # Learning rate annealing
    # ========================================================

    if args.anneal_lr:

        frac = (
            1.0
            -
            (iteration - 1.0)
            /
            args.num_iterations
        )

        lrnow = frac * args.learning_rate

        optimizer.param_groups[0]["lr"] = lrnow


    # ========================================================
    # Collect experience
    # ========================================================

    for step in range(
        args.num_steps
    ):

        global_step += args.num_envs

        obs[step] = next_obs

        dones[step] = next_done


        # ----------------------------------------------------
        # Select action
        # ----------------------------------------------------

        with torch.no_grad():

            (
                action,
                logprob,
                _,
                value
            ) = agent.get_action_and_value(
                next_obs
            )

            values[step] = value.flatten()


        actions[step] = action

        logprobs[step] = logprob


        # ----------------------------------------------------
        # Execute action
        # ----------------------------------------------------

        (
            next_obs_np,
            reward,
            terminations,
            truncations,
            infos
        ) = envs.step(
            action.cpu().numpy()
        )


        # ----------------------------------------------------
        # Episode information
        # ----------------------------------------------------

        next_done_np = np.logical_or(
            terminations,
            truncations
        )


        if "episode" in infos:

            for i in range(
                args.num_envs
            ):

                if infos["_episode"][i]:

                    episodic_return = (
                        infos["episode"]["r"][i]
                    )

                    episodic_length = (
                        infos["episode"]["l"][i]
                    )

                    print(
                        f"global_step={global_step}, "
                        f"env={i}, "
                        f"episodic_return="
                        f"{episodic_return}, "
                        f"episodic_length="
                        f"{episodic_length}"
                    )


                    writer.add_scalar(
                        "charts/episodic_return",
                        episodic_return,
                        global_step
                    )


                    writer.add_scalar(
                        "charts/episodic_length",
                        episodic_length,
                        global_step
                    )


        # ----------------------------------------------------
        # Save reward
        # ----------------------------------------------------

        rewards[step] = torch.tensor(
            reward
        ).to(device).view(-1)


        # ----------------------------------------------------
        # Prepare next state
        # ----------------------------------------------------

        next_obs = torch.Tensor(
            next_obs_np
        ).to(device)


        next_done = torch.Tensor(
            next_done_np
        ).to(device)


    # ========================================================
    # GAE
    # ========================================================

    with torch.no_grad():

        next_value = agent.get_value(
            next_obs
        ).reshape(1, -1)


        advantages = torch.zeros_like(
            rewards
        ).to(device)


        lastgaelam = 0


        for t in reversed(
            range(args.num_steps)
        ):

            # ------------------------------------------------
            # Determine next value
            # ------------------------------------------------

            if t == args.num_steps - 1:

                nextnonterminal = (
                    1.0 - next_done
                )

                nextvalues = next_value

            else:

                nextnonterminal = (
                    1.0 - dones[t + 1]
                )

                nextvalues = values[t + 1]


            # ------------------------------------------------
            # TD Error / Delta
            # ------------------------------------------------

            delta = (

                rewards[t]

                +

                args.gamma
                * nextvalues
                * nextnonterminal

                -

                values[t]

            )


            # ------------------------------------------------
            # GAE
            # ------------------------------------------------

            lastgaelam = (

                delta

                +

                args.gamma
                * args.gae_lambda
                * nextnonterminal
                * lastgaelam

            )


            advantages[t] = lastgaelam


        # ----------------------------------------------------
        # Returns
        # ----------------------------------------------------

        returns = advantages + values


    # ========================================================
    # Debug first iteration
    # ========================================================

    if iteration == 1:

        print("\n========== FIRST PPO BATCH ==========")

        print("Rewards:")
        print(rewards[:, 0])

        print("\nValues:")
        print(values[:, 0])

        print("\nAdvantages:")
        print(advantages[:, 0])

        print("\nReturns:")
        print(returns[:, 0])

        print("=====================================\n")


    # ========================================================
    # Flatten batch
    # ========================================================

    b_obs = obs.reshape(
        (-1,)
        +
        envs.single_observation_space.shape
    )


    b_logprobs = logprobs.reshape(-1)


    b_actions = actions.reshape(
        (-1,)
        +
        envs.single_action_space.shape
    )


    b_advantages = advantages.reshape(-1)


    b_returns = returns.reshape(-1)


    b_values = values.reshape(-1)


    # ========================================================
    # PPO Optimization
    # ========================================================

    b_inds = np.arange(
        args.batch_size
    )

    clipfracs = []


    for epoch in range(
        args.update_epochs
    ):

        np.random.shuffle(
            b_inds
        )


        for start in range(
            0,
            args.batch_size,
            args.minibatch_size
        ):

            end = (
                start
                +
                args.minibatch_size
            )


            mb_inds = b_inds[
                start:end
            ]


            # ------------------------------------------------
            # New policy evaluation
            # ------------------------------------------------

            (
                _,
                newlogprob,
                entropy,
                newvalue
            ) = agent.get_action_and_value(

                b_obs[mb_inds],

                b_actions.long()[mb_inds]

            )


            logratio = (
                newlogprob
                -
                b_logprobs[mb_inds]
            )


            ratio = logratio.exp()


            # ------------------------------------------------
            # KL + clipping statistics
            # ------------------------------------------------

            with torch.no_grad():

                old_approx_kl = (
                    -logratio
                ).mean()


                approx_kl = (
                    (ratio - 1)
                    -
                    logratio
                ).mean()


                clipfracs.append(

                    (
                        (ratio - 1.0).abs()
                        >
                        args.clip_coef
                    )
                    .float()
                    .mean()
                    .item()

                )


            # ------------------------------------------------
            # Advantages
            # ------------------------------------------------

            mb_advantages = (
                b_advantages[mb_inds]
            )


            if args.norm_adv:

                mb_advantages = (

                    mb_advantages
                    -
                    mb_advantages.mean()

                ) / (

                    mb_advantages.std()
                    +
                    1e-8

                )


            # =================================================
            # Policy Loss
            # =================================================

            pg_loss1 = (

                -mb_advantages
                *
                ratio

            )


            pg_loss2 = (

                -mb_advantages
                *
                torch.clamp(

                    ratio,

                    1
                    -
                    args.clip_coef,

                    1
                    +
                    args.clip_coef

                )

            )


            pg_loss = torch.max(

                pg_loss1,
                pg_loss2

            ).mean()


            # =================================================
            # Value Loss
            # =================================================

            newvalue = newvalue.view(-1)


            if args.clip_vloss:

                v_loss_unclipped = (

                    newvalue
                    -
                    b_returns[mb_inds]

                ) ** 2


                v_clipped = (

                    b_values[mb_inds]

                    +

                    torch.clamp(

                        newvalue
                        -
                        b_values[mb_inds],

                        -args.clip_coef,

                        args.clip_coef

                    )

                )


                v_loss_clipped = (

                    v_clipped
                    -
                    b_returns[mb_inds]

                ) ** 2


                v_loss_max = torch.max(

                    v_loss_unclipped,

                    v_loss_clipped

                )


                v_loss = (
                    0.5
                    *
                    v_loss_max.mean()
                )

            else:

                v_loss = (

                    0.5
                    *
                    (
                        newvalue
                        -
                        b_returns[mb_inds]
                    ) ** 2
                ).mean()


            # =================================================
            # Entropy
            # =================================================

            entropy_loss = entropy.mean()


            # =================================================
            # Total Loss
            # =================================================

            loss = (

                pg_loss

                -

                args.ent_coef
                *
                entropy_loss

                +

                v_loss
                *
                args.vf_coef

            )


            # =================================================
            # Backpropagation
            # =================================================

            optimizer.zero_grad()

            loss.backward()


            nn.utils.clip_grad_norm_(
                agent.parameters(),
                args.max_grad_norm
            )


            optimizer.step()


        # ----------------------------------------------------
        # Target KL
        # ----------------------------------------------------

        if (

            args.target_kl is not None

            and

            approx_kl > args.target_kl

        ):

            break


    # ========================================================
    # Explained Variance
    # ========================================================

    y_pred = (
        b_values
        .detach()
        .cpu()
        .numpy()
    )


    y_true = (
        b_returns
        .detach()
        .cpu()
        .numpy()
    )


    var_y = np.var(
        y_true
    )


    explained_var = (

        np.nan

        if var_y == 0

        else

        1
        -
        np.var(y_true - y_pred)
        /
        var_y

    )


    # ========================================================
    # TensorBoard logging
    # ========================================================

    writer.add_scalar(
        "charts/learning_rate",
        optimizer.param_groups[0]["lr"],
        global_step
    )


    writer.add_scalar(
        "losses/value_loss",
        v_loss.item(),
        global_step
    )


    writer.add_scalar(
        "losses/policy_loss",
        pg_loss.item(),
        global_step
    )


    writer.add_scalar(
        "losses/entropy",
        entropy_loss.item(),
        global_step
    )


    writer.add_scalar(
        "losses/old_approx_kl",
        old_approx_kl.item(),
        global_step
    )


    writer.add_scalar(
        "losses/approx_kl",
        approx_kl.item(),
        global_step
    )


    writer.add_scalar(
        "losses/clipfrac",
        np.mean(clipfracs),
        global_step
    )


    writer.add_scalar(
        "losses/explained_variance",
        explained_var,
        global_step
    )


    # ========================================================
    # SPS
    # ========================================================

    sps = int(
        global_step
        /
        (time.time() - start_time)
    )


    print(
        f"Iteration {iteration}/"
        f"{args.num_iterations} | "
        f"global_step={global_step} | "
        f"SPS={sps}"
    )


    writer.add_scalar(
        "charts/SPS",
        sps,
        global_step
    )


# ============================================================
# 14. Finish
# ============================================================

envs.close()

writer.close()

print("\n================================")
print("TRAINING FINISHED")
print("================================")
print("TensorBoard run:")
print(f"runs/{run_name}")

Batch size: 512
Minibatch size: 128
Number of iterations: 19
Device: cuda
GPU: NVIDIA GeForce GTX 1650
global_step=52, env=0, episodic_return=13.0, episodic_length=13
global_step=56, env=1, episodic_return=14.0, episodic_length=14
global_step=76, env=2, episodic_return=19.0, episodic_length=19
global_step=96, env=3, episodic_return=24.0, episodic_length=24
global_step=100, env=1, episodic_return=10.0, episodic_length=10
global_step=140, env=2, episodic_return=15.0, episodic_length=15
global_step=164, env=1, episodic_return=15.0, episodic_length=15
global_step=212, env=1, episodic_return=11.0, episodic_length=11
global_step=240, env=3, episodic_return=35.0, episodic_length=35
global_step=264, env=0, episodic_return=52.0, episodic_length=52
global_step=288, env=2, episodic_return=36.0, episodic_length=36
global_step=356, env=1, episodic_return=35.0, episodic_length=35
global_step=360, env=3, episodic_return=29.0, episodic_length=29
global_step=388, env=0, episodic_return=30.0, episodic_l